In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"

In [3]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,  
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [11]:
import json

In [34]:
def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
    "format": "json" or "python" or "regex"
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [35]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [37]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""
    
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

In [40]:
def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
    You are an expert code reviewer. Your task is to evaluate the AI-generated solution.
    
    Original Task:
    <task>
    {test_case["task"]}
    </task>

    Solution to Evaluate:
    <solution>
    {output}
    </solution>
    
    Output Format
    Provide your evaluation as a structured JSON object with the following fields:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10

    Respond with JSON. Keep your response concise and direct.
    Example response shape:
    {{
        "strengths": string[],
        "weaknesses": string[],
        "reasoning": string,
        "score": number
    }}
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [33]:
import re
import ast

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0
    
def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [38]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    print("score:", model_score)
    print("reasoning:", reasoning)

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [22]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [41]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

score: 7
reasoning: The solution correctly solves the main problem of extracting resource types from standard AWS ARNs. It properly handles both simple resource types and those with forward slash separators. However, it's missing robust input validation and ARN format verification that would make it more production-ready.
score: 8
reasoning: The solution correctly addresses all core requirements with proper IAM policy syntax and resource targeting. The separation of bucket and object permissions is architecturally sound. While functional as-is, it could be enhanced with additional security considerations for production use.
score: 7
reasoning: The regex correctly implements most AWS S3 bucket naming rules including character restrictions, boundary conditions, and consecutive dot prevention. However, there's a subtle length validation error where the quantifier {1,61} in the middle section creates an effective range of 3-63 characters when AWS allows up to 63 characters, potentially rej

In [42]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\ndef extract_resource_type(arn):\n    parts = arn.split(':')\n    if len(parts) >= 6:\n        resource_part = parts[5]\n        if '/' in resource_part:\n            return resource_part.split('/')[0]\n        else:\n            return resource_part\n    return None\n",
    "test_case": {
      "task": "Create a Python function that takes an AWS ARN string and extracts the resource type (e.g., 'user', 'role', 'bucket') from it. The function should handle standard ARN format: arn:partition:service:region:account-id:resource-type/resource-name",
      "format": "python"
    },
    "score": 8.5,
    "reasoning": "The solution correctly solves the main problem of extracting resource types from standard AWS ARNs. It properly handles both simple resource types and those with forward slash separators. However, it's missing robust input validation and ARN format verification that would make it more production-ready."
  },
  {
    "output": "\n{\n  \"Version\": \"2012-10-